En este notebook vamos a hacer una simulación del mes de febrero con nuestra estimación de estaciones (y sus respectivos tamaños).

In [35]:
import pandas as pd

estaciones = pd.read_csv('centroides2.csv')
viajes = pd.read_csv('potenciales_viajes_cubiertos.csv')
tamaño_estaciones = pd.read_csv('tamaño_cluster.csv')

viajes['tsO'] = pd.to_datetime(viajes['tsO'])
viajes['tsD'] = pd.to_datetime(viajes['tsD'])

In [2]:
viajes.head()

,Unnamed: 0,idS,tsO,tsD,price,tt,dis,vel,lonO,latO,lonD,latD,duration_calc,clusterOrigen,clusterFinal,origin_lat,origin_lon,destination_lat,destination_lon
0,0,A0H4,2021-02-03 18:10:03,2021-02-03 18:17:44,2.1525,461,1715.336751,13.395254,12.466222,41.867388,12.470660,41.853908,461.0,174.0,122.0,NaN,NaN,NaN,NaN
1,1,A0H4,2021-02-13 18:21:13,2021-02-13 18:25:33,1.6500,260,1234.472044,17.092690,12.471143,41.923692,12.467502,41.934306,260.0,59.0,98.0,NaN,NaN,NaN,NaN
2,2,A0H4,2021-02-14 13:39:54,2021-02-14 13:48:03,2.2225,489,2221.481536,16.354465,12.467524,41.934342,12.486330,41.928270,489.0,98.0,137.0,NaN,NaN,NaN,NaN
3,3,A0H4,2021-02-14 14:37:53,2021-02-14 14:57:53,4.0000,1200,4562.843566,13.688531,12.486275,41.928301,12.457922,41.904302,1200.0,137.0,113.0,NaN,NaN,NaN,NaN
4,4,A0H4,2021-02-15 13:31:24,2021-02-15 13:34:45,1.5025,201,550.154792,9.853519,12.457876,41.904303,12.460773,41.907606,201.0,113.0,179.0,NaN,NaN,NaN,NaN


In [17]:
dia = 11
viajes_11 = viajes[(viajes['tsO'] > f'2021-02-{dia} 00:00:00') & (viajes['tsO'] < f'2021-02-{dia} 23:59:59')][['idS', 'tsO', 'tsD', 'price', 'clusterOrigen', 'clusterFinal']]
viajes_11['origen'] = viajes_11['clusterOrigen'].apply(lambda x: (estaciones[estaciones['cluster'] == x]['lat'].item(), estaciones[estaciones['cluster'] == x]['lon'].item()))
viajes_11['destino'] = viajes_11['clusterFinal'].apply(lambda x: (estaciones[estaciones['cluster'] == x]['lat'].item(), estaciones[estaciones['cluster'] == x]['lon'].item()))
viajes_11['tsO'] = pd.to_datetime(viajes_11['tsO'])
viajes_11['tsD'] = pd.to_datetime(viajes_11['tsD'])
viajes_11

,idS,tsO,tsD,price,clusterOrigen,clusterFinal,origen,destino
58,A0N7,2021-02-11 08:54:33,2021-02-11 09:19:03,4.6750,46.0,149.0,"(41.8773378, 12.5307418)","(41.9072608, 12.4732515)"
82,A0P6,2021-02-11 14:36:54,2021-02-11 14:48:43,2.7725,34.0,165.0,"(41.9119802, 12.4755149)","(41.9008112, 12.5000502)"
186,A2E3,2021-02-11 08:46:34,2021-02-11 09:01:14,3.2000,55.0,118.0,"(41.9319168, 12.5203835)","(41.9084313, 12.5028896)"
206,A2M3,2021-02-11 15:41:23,2021-02-11 15:53:13,2.7750,133.0,149.0,"(41.9049253, 12.4441233)","(41.9072608, 12.4732515)"
252,A2R1,2021-02-11 17:21:34,2021-02-11 17:43:24,4.2750,122.0,147.0,"(41.8573232, 12.4725779)","(41.8706408, 12.4684708)"
...,...,...,...,...,...,...,...,...
20200,Z8J1,2021-02-11 19:29:23,2021-02-11 19:36:43,2.1000,157.0,58.0,"(41.8462101, 12.4826066)","(41.845972, 12.4904363)"
20219,Z8Y2,2021-02-11 09:55:44,2021-02-11 10:02:53,2.0725,9.0,185.0,"(41.8827447, 12.5202617)","(41.8925043, 12.5092605)"
20260,Z9N9,2021-02-11 14:17:33,2021-02-11 14:42:04,4.6775,87.0,129.0,"(41.8875346, 12.5213157)","(41.8934018, 12.4871321)"
20287,Z9U3,2021-02-11 11:31:43,2021-02-11 11:39:53,2.2250,171.0,74.0,"(41.8861783, 12.5172333)","(41.8728698, 12.5279662)"


In [5]:
tamaño_estaciones

,Station,Tamaño
0,0.0,10
1,1.0,11
2,2.0,8
3,3.0,8
4,4.0,9
...,...,...
185,185.0,7
186,186.0,7
187,187.0,4
188,188.0,5


In [51]:
import pandas as pd
import numpy as np

def calcular_optimos_por_periodos(viajes_mes, estaciones_df, fechas_redistribucion):
    """
    Calcula el estado inicial óptimo para el INICIO de cada periodo de redistribución.
    El cálculo considera la acumulación de demanda durante todo el periodo (ej. una semana).
    """

    # 1. Preparar Flujos (Igual que antes)
    salidas = viajes_mes[['tsO', 'clusterOrigen']].copy()
    salidas.columns = ['time', 'station_id']
    salidas['change'] = -1

    llegadas = viajes_mes[['tsD', 'clusterFinal']].copy()
    llegadas.columns = ['time', 'station_id']
    llegadas['change'] = 1

    flujo_total = pd.concat([salidas, llegadas]).sort_values('time')

    # 2. ASIGNAR PERIODOS
    # Convertimos las fechas de corte a datetime para comparar
    fechas_corte = pd.to_datetime(fechas_redistribucion)

    # Usamos searchsorted para ver en qué 'cajón' (periodo) cae cada viaje
    # Esto asigna 0 al primer intervalo, 1 al segundo, etc.
    # Nota: Asegúrate de que 'time' esté ordenado para searchsorted, o usa apply (más lento)
    # Aquí usamos un método vectorial robusto:
    flujo_total['period_start_date'] = pd.cut(
        flujo_total['time'],
        bins=list(fechas_corte) + [pd.Timestamp.max],
        labels=fechas_redistribucion,
        right=False
    )

    # Eliminamos viajes que queden fuera de los rangos (si los hay)
    flujo_total = flujo_total.dropna(subset=['period_start_date'])

    optimos_por_periodo = {} # Key: Fecha de inicio del periodo, Value: Dict de estaciones

    # 3. Iteramos por PERIODO (ej: Semana 1, Semana 2...)
    grupos_periodo = flujo_total.groupby('period_start_date')

    print(f"Calculando optimización para {len(grupos_periodo)} periodos de redistribución...")

    for fecha_inicio, flujo_periodo in grupos_periodo:

        optimum_states = {}
        grupos_estacion = flujo_periodo.groupby('station_id')

        for _, row in estaciones_df.iterrows():
            st_id = int(row['Station'])
            capacidad = row['Tamaño']
            col_name = f'station{st_id}'

            if st_id not in grupos_estacion.groups:
                optimum_states[col_name] = (capacidad, int(capacidad / 2))
                continue

            df_st = grupos_estacion.get_group(st_id).sort_values('time')

            # --- LA CLAVE ---
            # El cumsum ahora recorre VARIOS DÍAS.
            # Si el lunes pierdes 2 bicis y el martes pierdes 3, el acumulado llega a -5.
            cumsum = df_st['change'].cumsum()

            min_reach = cumsum.min()
            max_reach = cumsum.max()

            # Lógica de optimización idéntica, pero aplicada a la curva semanal
            needed_start = abs(min_reach) if min_reach < 0 else 0

            peak_occupancy = needed_start + max_reach
            final_start = needed_start

            # El conflicto es MUCHO más probable en periodos largos
            if peak_occupancy > capacidad:
                mid_flow = (max_reach + min_reach) / 2
                final_start = (capacidad / 2) - mid_flow

            final_start = max(0, min(capacidad, np.round(final_start)))
            optimum_states[col_name] = (capacidad, final_start)

        optimos_por_periodo[fecha_inicio] = optimum_states

    return optimos_por_periodo

# --- CONFIGURACIÓN ---
# Definimos los lunes de febrero como días de redistribución
dias_redistribucion = ['2021-02-01', '2021-02-08', '2021-02-15', '2021-02-22']

# Calculamos
optimos_semanales = calcular_optimos_por_periodos(viajes, tamaño_estaciones, dias_redistribucion)

/tmp/ipython-input-1442920809.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grupos_periodo = flujo_total.groupby('period_start_date')


Calculando optimización para 4 periodos de redistribución...


In [52]:
optimos_semanales

{'2021-02-01': {'station0': (np.float64(10.0), 0),
  'station1': (np.float64(11.0), np.int64(8)),
  'station2': (np.float64(8.0), np.int64(7)),
  'station3': (np.float64(8.0), np.int64(6)),
  'station4': (np.float64(9.0), np.int64(3)),
  'station5': (np.float64(8.0), np.float64(6.0)),
  'station6': (np.float64(9.0), np.float64(2.0)),
  'station7': (np.float64(4.0), np.float64(3.0)),
  'station8': (np.float64(6.0), 0),
  'station9': (np.float64(6.0), 0),
  'station10': (np.float64(12.0), np.float64(12.0)),
  'station11': (np.float64(8.0), np.int64(4)),
  'station12': (np.float64(13.0), np.int64(11)),
  'station13': (np.float64(4.0), np.float64(4.0)),
  'station14': (np.float64(8.0), 0),
  'station15': (np.float64(9.0), 0),
  'station16': (np.float64(7.0), 0),
  'station17': (np.float64(9.0), np.int64(5)),
  'station18': (np.float64(7.0), np.float64(3.0)),
  'station19': (np.float64(5.0), np.int64(1)),
  'station20': (np.float64(11.0), np.float64(11.0)),
  'station21': (np.float64(12.0),

In [57]:
count = {}
for semana in optimos_semanales.keys():
  if semana not in count.keys():
    count[semana] = 0
  for estacion in optimos_semanales[semana]:
      count[semana] += optimos_semanales[semana][estacion][1]

count

{'2021-02-01': np.float64(624.0),
 '2021-02-08': np.float64(628.0),
 '2021-02-15': np.float64(645.0),
 '2021-02-22': np.float64(667.0)}

In [38]:
import numpy as np
import pandas as pd

start_time = f'2021-02-01 00:00:00'
end_time = f'2021-02-28 23:59:00'
minutos = pd.date_range(start=start_time, end=end_time, freq='min')

station_columns = [f'station{int(s)}' for s in tamaño_estaciones['Station']]
simulations_df = pd.DataFrame(index=minutos, columns=station_columns)

initial_states_dict = {
    f'station{int(row["Station"])}': (row['Tamaño'], np.round(row['Tamaño']/2))
    for index, row in tamaño_estaciones.iterrows()
}

simulations_df.loc[minutos[0]] = initial_states_dict
saltar = []

errores_df = []
for i, minuto in enumerate(minutos[1:]):
    # Copy previous minute's state
    simulations_df.loc[minuto] = simulations_df.loc[minutos[i]]

    # Trips starting at this minute
    trips_starting_now = viajes[viajes['tsO'].dt.floor('min') == minuto]
    for index, row in trips_starting_now.iterrows():
        origen_cluster = int(row['clusterOrigen'])
        station_col = f'station{origen_cluster}'
        if station_col in simulations_df.columns:
            current_capacity, current_occupation = simulations_df.loc[minuto, station_col]
            if current_occupation == 0:
              errores_df.append({'Error': 'Falta bicis', 'Station': station_col})
              saltar.append(index)
              simulations_df.loc[minuto, station_col] = (current_capacity, max(0.0, current_occupation))
            else:
              simulations_df.loc[minuto, station_col] = (current_capacity, max(0.0, current_occupation - 1.0))

    # Trips ending at this minute
    trips_ending_now = viajes[viajes['tsD'].dt.floor('min') == minuto]
    for index, row in trips_ending_now.iterrows():
      if index in saltar:
        continue
      else:
        destino_cluster = int(row['clusterFinal'])
        station_col = f'station{destino_cluster}'
        if station_col in simulations_df.columns:
            current_capacity, current_occupation = simulations_df.loc[minuto, station_col]
            if current_occupation == current_capacity:
              errores_df.append({'Error': 'Falta espacio', 'Station': station_col})
              simulations_df.loc[minuto, station_col] = (current_capacity, min(current_capacity, current_occupation))
            else:
              simulations_df.loc[minuto, station_col] = (current_capacity, min(current_capacity, current_occupation + 1.0))

errores_df = pd.DataFrame(errores_df)
print("Simulation completed for the day.")
errores_df

Simulation completed for the day.


,Error,Station
0,Falta espacio,station189
1,Falta bicis,station139
2,Falta bicis,station139
3,Falta bicis,station139
4,Falta bicis,station95
...,...,...
5766,Falta bicis,station21
5767,Falta bicis,station43
5768,Falta bicis,station23
5769,Falta bicis,station171
